# Beca 18 RAG Chatbot
**Retrieval-Augmented Generation sobre el Reglamento Oficial | PRONABEC 2026**

Pipeline que responde preguntas sobre Beca 18 exclusivamente desde el documento oficial, rechazando consultas fuera de tema.

In [ ]:
# Run only in Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(
        ['pip', 'install', '-q',
         'pypdf', 'tiktoken', 'langchain-text-splitters',
         'google-genai', 'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv'],
        check=True,
    )
    print('Dependencies installed.')

---
## Step 0 | Environment Setup & Gemini Client

In [ ]:
import os
import time
import random
from pathlib import Path

from dotenv import load_dotenv

NL = chr(10)  # newline — avoids \n in string literals throughout the notebook

load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise EnvironmentError(
        'GEMINI_API_KEY not found. '
        'Create a .env file with: GEMINI_API_KEY=your_key_here'
    )

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
print('Gemini client initialized.')

# Paths — works both locally (notebooks/) and in Colab
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
CHROMA_PATH  = str(PROJECT_ROOT / 'chroma_db_beca18')

# Accept any PDF in data/ (filename may vary by download source)
pdf_candidates = list(DATA_DIR.glob('*.pdf'))
assert pdf_candidates, 'No PDF found in data/. Download the Beca 18 regulation PDF and place it there.'
PDF_PATH = pdf_candidates[0]

print(f'PDF path  : {PDF_PATH}')
print(f'ChromaDB  : {CHROMA_PATH}')
print('PDF found.')

---
## Step 1 | PDF Extraction with `[PAGE N]` Markers
Extract text page-by-page with `pypdf`, clean whitespace, and report character/word counts.

In [ ]:
from pypdf import PdfReader


def clean_page_text(text):
    """Strip redundant whitespace and lone page-number lines."""
    lines = [line.strip() for line in text.splitlines()]
    lines = [l for l in lines if l and not l.isdigit()]
    return NL.join(lines)


reader      = PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f'Total pages: {total_pages}')

pages_data      = []  # list of {page: int, text: str}
full_text_parts = []  # for global stats

for i, page in enumerate(reader.pages):
    raw     = page.extract_text() or ''
    cleaned = clean_page_text(raw)
    pages_data.append({'page': i + 1, 'text': cleaned})
    full_text_parts.append('[PAGE ' + str(i + 1) + ']' + NL + cleaned)

full_text = (NL + NL).join(full_text_parts)

print(f'Character count : {len(full_text):,}')
print(f'Word count      : {len(full_text.split()):,}')
print(NL + '--- Preview (first 500 chars) ---')
print(full_text[:500])

---
## Step 2 | Token Counting, Chunking Justification & Splitting
Count tokens with `tiktoken` (cl100k_base), justify 400-token chunks / 60-token overlap, split with metadata.

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.notebook import tqdm as tqdm_nb

enc          = tiktoken.get_encoding('cl100k_base')
total_tokens = len(enc.encode(full_text))

print(f'Total tokens (cl100k_base) : {total_tokens:,}')
print(f'Embedding model limit      : 8,192 tokens')
print(f'Chunk size chosen          : 400 tokens  ({400 / 8192 * 100:.1f}% of limit)')
print(f'Overlap                    : 60 tokens   (maintains context at boundaries)')
print(f'Estimated num chunks       : ~{total_tokens // (400 - 60)}')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    length_function=lambda t: len(enc.encode(t)),
    separators=[chr(10) + chr(10), chr(10), '. ', ' ', ''],
)

all_chunks = []
for page_data in tqdm_nb(pages_data, desc='Chunking pages'):
    if not page_data['text'].strip():
        continue
    page_chunks = splitter.create_documents(
        texts=[page_data['text']],
        metadatas=[{
            'document': 'beca18_reglamento',
            'topic':    'beca18',
            'language': 'es',
            'page':     page_data['page'],
        }],
    )
    all_chunks.extend(page_chunks)

print(f'Total chunks generated : {len(all_chunks)}')
print(f'Sample metadata        : {all_chunks[0].metadata}')
print(f'Sample content         :\n{all_chunks[0].page_content[:250]}')

---
## Step 3 | Embedding Functions with Exponential Backoff
`embed_documents()` uses `RETRIEVAL_DOCUMENT`; `embed_query()` uses `RETRIEVAL_QUERY`. Both handle rate limits (~60 req/min free tier).

In [ ]:
def embed_documents(texts):
    """Embed a list of document texts using RETRIEVAL_DOCUMENT task type."""
    embeddings = []
    for text in tqdm_nb(texts, desc='Embedding docs'):
        for attempt in range(5):
            try:
                resp = client.models.embed_content(
                    model='gemini-embedding-001',
                    contents=text,
                    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT'),
                )
                embeddings.append(list(resp.embeddings[0].values))
                break
            except Exception as exc:
                msg = str(exc).lower()
                if '429' in msg or 'quota' in msg or 'rate' in msg:
                    wait = (2 ** attempt) + random.uniform(0.1, 0.5)
                    print(f'  Rate limit (attempt {attempt + 1}/5), waiting {wait:.1f}s...')
                    time.sleep(wait)
                else:
                    raise
        time.sleep(1.1)  # stay safely under 60 req/min
    return embeddings


def embed_query(text):
    """Embed a single query string using RETRIEVAL_QUERY task type."""
    for attempt in range(5):
        try:
            resp = client.models.embed_content(
                model='gemini-embedding-001',
                contents=text,
                config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY'),
            )
            return list(resp.embeddings[0].values)
        except Exception as exc:
            msg = str(exc).lower()
            if '429' in msg or 'quota' in msg or 'rate' in msg:
                wait = (2 ** attempt) + random.uniform(0.1, 0.5)
                print(f'Rate limit (attempt {attempt + 1}/5), waiting {wait:.1f}s...')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('embed_query: all 5 retry attempts failed.')


# Smoke test
test_vec = embed_query('prueba de embedding')
print(f'Embedding dimension : {len(test_vec)}')
print(f'First 5 values      : {[round(v, 6) for v in test_vec[:5]]}')

---
## Step 4 | Persistent ChromaDB Index (Idempotent)
Create or reuse a persistent ChromaDB collection (cosine distance). Skip embedding if documents already exist.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name='beca18',
    metadata={'hnsw:space': 'cosine'},
)

if collection.count() > 0:
    print(f'Collection already has {collection.count()} chunks. Skipping indexing.')
else:
    print(f'Indexing {len(all_chunks)} chunks — may take several minutes on the free tier...')

    texts = [c.page_content for c in all_chunks]
    metas = [c.metadata     for c in all_chunks]
    ids   = ['chunk_' + str(i) for i in range(len(all_chunks))]

    embeddings = embed_documents(texts)

    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metas,
        ids=ids,
    )
    print(f'Done. {collection.count()} chunks indexed.')

---
## Step 5 | `semantic_search` Function
Retrieve the k most relevant chunks via cosine similarity. Test with a sample query.

In [ ]:
def semantic_search(question, k=5):
    """Return top-k chunks most relevant to question."""
    q_emb   = embed_query(question)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {'text': doc, 'metadata': meta, 'distance': dist}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0],
        )
    ]


SAMPLE = '¿Cuáles son los requisitos para postular a Beca 18?'
hits   = semantic_search(SAMPLE, k=5)
print('Query: ' + SAMPLE)
print('Retrieved: ' + str(len(hits)) + ' chunks' + NL)
for i, h in enumerate(hits[:3], 1):
    page = h['metadata'].get('page')
    dist = h['distance']
    print(f'--- Result {i} | Page {page} | dist={dist:.4f} ---')
    print(h['text'][:280])
    print()

---
## Step 6 | `answer_with_context` + 6 Test Questions
Grounded answers via `gemini-2.5-flash` with strict system prompt. 5 on-topic + 1 off-topic.

In [ ]:
SYSTEM_PROMPT = NL.join([
    'Eres un asistente especializado en el reglamento del programa Beca 18 de PRONABEC (Peru).',
    '',
    'REGLAS ESTRICTAS:',
    '1. Responde UNICAMENTE basandote en los fragmentos de contexto proporcionados. No uses conocimiento externo.',
    '2. Cita siempre el numero de pagina de donde proviene la informacion (ej: Segun la pagina 5...).',
    '3. Si la respuesta no esta en el contexto, di exactamente: No encontre informacion sobre eso en el reglamento de Beca 18.',
    '4. Si la pregunta NO esta relacionada con Beca 18, di exactamente: Esta pregunta esta fuera del ambito del reglamento de Beca 18.',
    '5. Responde siempre en espanol.',
])


def answer_with_context(question, k=5):
    """Retrieve context chunks and generate a grounded answer via gemini-2.5-flash."""
    sources = semantic_search(question, k=k)

    context_blocks = []
    for i, s in enumerate(sources, 1):
        page  = s['metadata'].get('page', '?')
        text  = s['text']
        block = '[Fragmento ' + str(i) + ' | Pagina ' + str(page) + ']' + NL + text
        context_blocks.append(block)
    context = (NL + NL).join(context_blocks)

    user_prompt = (
        'Contexto del reglamento de Beca 18:' + NL + NL
        + context + NL + NL
        + 'Pregunta: ' + question
    )

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
        ),
        contents=user_prompt,
    )
    return {'answer': response.text, 'sources': sources}


# 6 test questions
TEST_CASES = [
    ('Elegibilidad', '¿Quienes son elegibles para postular a Beca 18?'),
    ('Modalidades',  '¿Cuales son las modalidades de la beca?'),
    ('Estipendio',   '¿Cual es el monto del estipendio mensual del becario?'),
    ('Obligaciones', '¿Cuales son las obligaciones del becario durante el programa?'),
    ('Perdida',      '¿Bajo que condiciones se pierde la beca?'),
    ('Off-topic',    '¿Cual es la capital de Francia?'),
]

SEP = '=' * 65
for label, question in TEST_CASES:
    print(SEP)
    print('  [' + label + ']  ' + question)
    print(SEP)
    result = answer_with_context(question, k=5)
    print(result['answer'])
    print()

---
## Step 7 | Interactive Chat UI (`ipywidgets`)
Text input · Ask/Clear buttons · k slider · response area · collapsible source accordion.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_input = widgets.Text(
    placeholder='Escribe tu pregunta sobre Beca 18...',
    layout=widgets.Layout(width='65%'),
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Chunks (k):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='32%'),
)
ask_button   = widgets.Button(description='Ask',   button_style='primary', icon='search')
clear_button = widgets.Button(description='Clear', button_style='warning', icon='times')
output_area  = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='12px', min_height='80px')
)


def on_ask(b):
    question = question_input.value.strip()
    with output_area:
        clear_output(wait=True)
        if not question:
            print('Por favor escribe una pregunta.')
            return
        display(widgets.HTML('<i>Consultando: ' + question + '</i>'))
        result      = answer_with_context(question, k=k_slider.value)
        answer_html = result['answer'].replace(chr(10), '<br>')
        display(widgets.HTML('<br><b>Respuesta:</b><br>' + answer_html))

        source_boxes = []
        for s in result['sources']:
            page   = s['metadata'].get('page', '?')
            dist   = s['distance']
            text   = s['text']
            header = '<b>Pagina ' + str(page) + ' | dist coseno: ' + f'{dist:.4f}' + '</b>'
            body   = '<pre style="white-space:pre-wrap;font-size:11px;background:#f8f8f8;padding:6px">' + text + '</pre>'
            source_boxes.append(widgets.HTML(header + body))

        n   = len(result['sources'])
        acc = widgets.Accordion(children=[widgets.VBox(source_boxes)])
        acc.set_title(0, 'Fuentes (' + str(n) + ' chunks recuperados)')
        acc.selected_index = None
        display(acc)


def on_clear(b):
    question_input.value = ''
    with output_area:
        clear_output()


ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

title = widgets.HTML(
    '<h3 style="margin-bottom:4px">Chatbot Beca 18 — Sistema RAG</h3>'
    '<p style="color:#555;margin-top:0">Responde exclusivamente desde el reglamento oficial de Beca 18 (PRONABEC 2026).</p>'
)
ui = widgets.VBox([
    title,
    widgets.HBox([question_input, ask_button, clear_button]),
    k_slider,
    output_area,
])
display(ui)